In [ ]:
print("========== SILVER PARAMETERS ==========")
print(f"start_date = '{start_date}'")
print("=======================================")

In [ ]:
'''df.write.format("delta").mode("overwrite").saveAsTable(
    "silver_events"
)'''

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

#start_date="2026-08-01"

# Read Bronze JSON
df = spark.read.option("multiline", "true").json(
    f"Files/{start_date}_earthquake_data.json"
)

# Select and flatten required columns
df = df.select(
    col("id").alias("earthquake_id"),
    col("geometry.coordinates").getItem(0).alias("longitude"),
    col("geometry.coordinates").getItem(1).alias("latitude"),
    col("geometry.coordinates").getItem(2).alias("elevation"),
    col("properties.title").alias("title"),
    col("properties.place").alias("place_description"),
    col("properties.sig").alias("sig"),
    col("properties.mag").alias("mag"),
    col("properties.magType").alias("magType"),
    col("properties.time").alias("time"),
    col("properties.updated").alias("updated")
)

# Convert Unix milliseconds → seconds → timestamp
from pyspark.sql.functions import col, from_unixtime

df = (
    df
    .withColumn("time", from_unixtime(col("time") / 1000).cast("timestamp"))
    .withColumn("updated", from_unixtime(col("updated") / 1000).cast("timestamp"))
)

# Target Silver Delta table
target = DeltaTable.forName(
    spark,
    "silver_events"
)

# Upsert into Silver
(
    target.alias("target")
    .merge(
        df.alias("source"),
        "target.earthquake_id = source.earthquake_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

'''
print("SILVER SOURCE SCHEMA AFTER MERGE:")
df.printSchema()
'''

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
print("========== SILVER INGESTION ==========")
print(f"Source file: Files/{start_date}_earthquake_data.json")
print(f"Records read from Bronze: {df.count()}")
print("=======================================")